# Load a processed dataset into Neo4j

Loads the artifacts of a processed dataset (made by `load_dataset.ipynb`) into Neo4j, so the dataset appears as a **project**.

- Requirements become `(:Requirement {id, text})`; everything else becomes `(:CodeEntity {id, text, kind})`. These are the labels C4 reads (see `shared/uskg/client.py`).
- Each node gets a `dataset` property; that is how the web app tells projects apart.
- **Idempotent:** it uses `MERGE` on `id`, so running it again is safe.
- The ground-truth links are deliberately **not** loaded. They belong only in the gold file; otherwise the agent could "find" them in the graph.

This is development seed data standing in for C1 and C2. **Do not run it against the team's shared graph.**
The settings cell prints which database it will write to. Check it before running the last cell.

**Needs:** `NEO4J_URI`, `NEO4J_USERNAME` and `NEO4J_PASSWORD` in `.env`.

In [ ]:
from pathlib import Path
import sys

# the notebook may be opened from scripts/ or from the project folder: find the project root either way
here = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [here, *here.parents] if (p / "agents").is_dir() and (p / "shared").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))
print("project root:", PROJECT_ROOT)

## Settings
Choose the dataset: `cm1`, `itrust`, `etour` or `libest`.

In [ ]:
import json
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

DATASET = "itrust"   # one of: cm1, itrust, etour, libest
ARTIFACTS = PROJECT_ROOT / "data" / "processed" / DATASET / "artifacts.json"

load_dotenv(PROJECT_ROOT / ".env")
URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
print("dataset  :", DATASET)
print("file     :", ARTIFACTS)
print("database :", URI)

## Queries

In [ ]:
_REQUIREMENTS = """
UNWIND $rows AS row
MERGE (n:Requirement {id: row.id})
SET n.text = row.text, n.dataset = row.dataset
"""

_CODE_ENTITIES = """
UNWIND $rows AS row
MERGE (n:CodeEntity {id: row.id})
SET n.text = row.text, n.kind = row.type, n.dataset = row.dataset
"""

## 1 · Read the file
Nothing is written yet.

In [ ]:
artifacts = json.loads(ARTIFACTS.read_text(encoding="utf-8"))
rows = [
    {"id": a["id"], "type": a["type"], "text": a["text"], "dataset": a["metadata"].get("dataset")}
    for a in artifacts
]
requirements = [r for r in rows if r["type"] == "requirement"]
others = [r for r in rows if r["type"] != "requirement"]
print(f"{len(requirements)} requirements and {len(others)} code or design entities will be upserted into {URI}")

## 2 · Write to Neo4j

In [ ]:
password = os.getenv("NEO4J_PASSWORD", "")
if not password:
    raise SystemExit("NEO4J_PASSWORD is empty; fill in .env first.")

with GraphDatabase.driver(URI, auth=(os.getenv("NEO4J_USERNAME", "neo4j"), password)) as driver:
    driver.execute_query(_REQUIREMENTS, rows=requirements)
    driver.execute_query(_CODE_ENTITIES, rows=others)
print(f"upserted {len(rows)} nodes into {URI}")